# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from graphrfibatching import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

In [3]:
df = pd.read_csv(f"../../../data/top30groups/LongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [6]:
geodata = ['longitude', 'latitude']
combined_geo = df.copy()
combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
combined_geo = combined_geo.drop(columns=geodata)

In [7]:
import ast

def to_tuple_if_needed(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val  # already a tuple

combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [8]:
torch.cuda.empty_cache()


In [9]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds = []
y_trues = []
logs = []

# Default config (from previous best)
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1500,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'batch_size': 256
}

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")

    data_list, y_gcn, y_nrf, non_geo_features, train_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
        combined_geo, label_index, continuous_col=col)

    # Train using default parameters
    best_acc, best_epoch, best_precision, best_recall, best_f1, y_pred_decoded, y_true_decoded, \
    best_precision_micro, best_recall_micro, best_f1_micro, best_precision_macro, best_recall_macro, best_f1_macro, \
    roc_auc_weighted, roc_auc_micro, roc_auc_macro, epoch_logs = train_joint(
        data_list, y_gcn, y_nrf, non_geo_features, train_mask, test_mask,
        default_args, row_to_node_index, index_to_label, verbose=True)


    y_preds.append(y_pred_decoded)
    y_trues.append(y_true_decoded)
    logs.append(epoch_logs)

    # Save performance metrics
    results_path = f"Results{partition}/Results_{col}_prediction"
    with open(results_path, "w") as f:
        f.write(f"Best acc: {best_acc:.4f} at epoch {best_epoch} for {col} prediction\n")
        f.write(f"Weighted Precision: {best_precision:.4f}, Recall: {best_recall:.4f}, F1: {best_f1:.4f}\n")
        f.write(f"Macro Precision: {best_precision_macro:.4f}, Recall: {best_recall_macro:.4f}, F1: {best_f1_macro:.4f}\n")
        f.write(f"Micro Precision: {best_precision_micro:.4f}, Recall: {best_recall_micro:.4f}, F1: {best_f1_micro:.4f}\n")
        f.write(f"AUROC Weighted: {roc_auc_weighted:.4f}, Micro: {roc_auc_micro:.4f}, Macro: {roc_auc_macro:.4f}\n")

    # Save epoch timings
    log_path = f"Results{partition}/epoch_logs_{col}_prediction"
    with open(log_path, "w") as f:
        f.write('\n'.join(f"{x:.4f}" for x in epoch_logs))



Training model for weaptype1 prediction...


/home/jovyan/MEX0512/GTD_2025/Codes/Baselines/GraphRfi_LongLatCombined/graphrfibatching.py:84: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data = Data(x=node_feat, edge_index=edge_index, idx=torch.tensor(row_to_node_index[i]))
Training epochs:   0%|          | 1/1500 [00:01<39:11,  1.57s/it]

Epoch 01 | GCN MSE Loss: 1.0420 | NRF Loss: 3.3674 | JOINT Loss: 4.4094 | NRF Acc: 0.4214


Training epochs:   0%|          | 2/1500 [00:02<33:58,  1.36s/it]

Epoch 02 | GCN MSE Loss: 0.9947 | NRF Loss: 3.2881 | JOINT Loss: 4.2829 | NRF Acc: 0.4471


...

Epoch 478 | GCN MSE Loss: 0.8338 | NRF Loss: 0.3915 | JOINT Loss: 1.2253 | NRF Acc: 0.9045


Training epochs:  32%|███▏      | 479/1500 [09:23<19:48,  1.16s/it]

Epoch 479 | GCN MSE Loss: 0.8321 | NRF Loss: 0.3911 | JOINT Loss: 1.2232 | NRF Acc: 0.9045


Training epochs:  32%|███▏      | 479/1500 [09:24<20:04,  1.18s/it]

Early stopping at epoch 479
Best acc/epoch: 0.9068, epoch 179


In [10]:
"""Best acc: 0.9068 at epoch 179 for weaptype1 prediction
Weighted Precision: 0.9098, Recall: 0.9068, F1: 0.9022
Macro Precision: 0.9177, Recall: 0.9005, F1: 0.9038
Micro Precision: 0.9068, Recall: 0.9068, F1: 0.9068
AUROC Weighted: 0.9979, Micro: 0.9978, Macro: 0.9979

"""

'Best acc: 0.7724 at epoch 179 for weaptype1 prediction\nWeighted Precision: 0.6840, Recall: 0.7724, F1: 0.7078\nMacro Precision: 0.6443, Recall: 0.6957, F1: 0.6526\nMicro Precision: 0.7724, Recall: 0.7724, F1: 0.7724\nAUROC Weighted: 0.9904, Micro: 0.9885, Macro: 0.9908\n'

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       1.00      0.92      0.96        37
        African National Congress (South Africa)       1.00      1.00      1.00        45
                                Al-Qaida in Iraq       0.92      0.85      0.88        52
        Al-Qaida in the Arabian Peninsula (AQAP)       0.93      0.88      0.91        77
                                      Al-Shabaab       0.97      1.00      0.99        36
             Basque Fatherland and Freedom (ETA)       1.00      0.89      0.94        55
                                      Boko Haram       0.80      0.87      0.83        45
  Communist Party of India - Maoist (CPI-Maoist)       0.97      0.92      0.94        37
       Corsican National Liberation Front (FLNC)       0.94      1.00      0.97        34
                       Donetsk People's Republic       1.00      0.96      0.98       110
Farabundo

In [12]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [13]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 100 to Results100/cm_100_weaptype1.png
